In [ ]:
from __future__ import annotations
import numpy as np
import random
import copy
import importlib

from typing import Tuple, List
from numpy import array, zeros
from matplotlib import pyplot as plt
from sklearn.preprocessing import normalize

# from Big_Class import Big_Class  # already imported one NETfuncs is imported
from User_Variables import User_Variables  # already imported one NETfuncs is imported
from Network_Structure import Network_Structure  # already imported one NETfuncs is imported
from Big_Class import Big_Class
from Network_State import Network_State
from Networkx_Net import Networkx_Net
from Color_Scheme import Color_Scheme
import matrix_functions, functions, statistics, plot_functions, figure_plots, colors

## colors

In [ ]:
importlib.reload(colors)

colors_lst, red, cmap = colors.color_scheme()
cmap

Colorscheme = Color_Scheme(show=True)

# Parameters

In [ ]:
## Parameters - stuff that you can confuse in different runs

## shuffle of training set
random_state = 52
    
## resistance-pressure proportionality factor
gamma: np.ndarray = np.array([1.0])
    
## method to update resistances - physical property of the system
# R_update: str = 'R_propto_dp'
# R_update: str = 'R_propto_sqrt_dp'
R_update: str = 'deltaR_propto_dp'
# R_update: str = 'deltaR_propto_dp_nonlin'
# R_update: str = 'R_propto_Q'
# R_update: str = 'deltaR_propto_Q'
# R_update: str = 'deltaR_propto_Power'
# R_update: str = 'R_propto_Power'
# R_update: str = 'R_propto_Q_exp'

# training_scheme = 'GD_like'
training_scheme = 'multip'

## use 1 or 2 sampled pressures at every time step
# use_p_tag: bool = True  
use_p_tag: bool = False

## how many loop iterations to stay under the same sampled p
stay_sample: int = 1  
# stay_sample: int = 100

## Use constant step size regardless of loss
normalize_step = False

normalize_M = True
normalize = 0.75

loss_type = 'MSE'

dK_step=10**(-8)

In [ ]:
## Parameters - can't be confused from runs

## task type
# task_type='Iris_classification'
task_type='Regression'

## input dataset type
# dataset_type = 'uniform_random'  # dataset sampled from uniform random distribution at each iteration
dataset_type = 'alternating ones'  # dataset is one at a single input and the rest zero, alternating between single inputs

## network type
# net_type='square'
net_type='FC'
# net_type='FC_connected_outputs'

# for square network
# net_height=8
# net_length=7
rand_seed=35  # seed for random nodes as inputs, outputs, ground
net_height=16
net_length=16

## specify # of nodes
Nin: int = 0
Ninter: int = 0
Nout: int = 0

in_nodes: np.ndarray(int) = array([], dtype=np.int_)
out_nodes: np.ndarray(int) = array([], dtype=np.int_)

# length of training dataset
iterations = 2800  # number of sampled of p
# hysteresis = 2.0
hysteresis = 1.0
# hysteresis = 0.0

if R_update in ['deltaR_propto_dp_nonlin', 'deltaR_propto_dp_nonlin_decay'] or hysteresis:
    normalize_loss = True
else:
    normalize_loss = False
    
normalize_loss = False  # temporarily force it

# measure accuracy every # steps
measure_accuracy_every = 15

supress_prints: bool = True  # whether to print information during training or not
# include_Power: bool = True
include_Power: bool = False

## Networkx sizes
scale: float = 50.0
squish: float = 0.01
    
## use a node with p=0 during all steps or not
add_ground=True

In [ ]:
access_interNodes: bool = False  # access and change pressure at interNodes (nodes between input and output) or not
noise_to_extra: bool = False  # add noise to extra outputs 

In [ ]:
def network_build_given_Nin_Nout(Nin: int, Nout: int, M_values: NDArray[np.float_]) -> tuple():
    
    # initialize Variables
    Variabs = User_Variables(iterations,\
                             Nin, \
                             Nout, \
                             gamma, \
                             R_update, \
                             training_scheme, \
                             use_p_tag, \
                             normalize_loss, \
                             supress_prints, \
                             task_type, \
                             dataset_type, \
                             measure_accuracy_every, \
                             normalize_step = normalize_step, \
                             hysteresis = hysteresis, \
                             anneal = anneal, \
                             T_annealing = T_annealing, \
                             Ninter = Ninter)
    
    Variabs.assign_alpha_vec(alpha)
    
    # Normalize M
    if normalize_M:
        ## Deal with normalizing task matrix M so task is concave
        M_values_norm = functions.normalize_M(M_values, normalize, Nin, Nout)
        Variabs.create_dataset_and_targets(random_state=random_state, M_values=M_values_norm, train_size = 0.8)
    else:
        Variabs.create_dataset_and_targets(random_state=random_state, M_values=M_values, train_size = 0.8)
    Variabs.create_noise_for_extras()
    BigClass = Big_Class(Variabs)

    ## Save color scheme in Big Class
    BigClass.add_Colors(Colorscheme)
        
    ## Assign input and output nodes a.f.o lattice size and row choice
    inOutGround_tuple = matrix_functions.build_input_output_and_ground(Variabs.Nin, Variabs.Nout, in_nodes=in_nodes, 
                                                                       out_nodes=out_nodes, add_ground=add_ground, 
                                                                       net_type=net_type, seed=rand_seed, net_height=net_height, 
                                                                       net_len=net_length)
        
    
    ## Structure class - build incidence matrices and 1d arrays of edges
    Strctr = Network_Structure(inOutGround_tuple, net_type, net_height, net_length)
    Strctr.build_incidence()
    BigClass.add_Strctr(Strctr)  # add to big class
    
    # initialize State  
    # R_vec_i = np.ones(Nin*Nout + Nin + Nout) + np.random.normal(loc=0.0, scale=0.01, size=Nin*Nout + Nin + Nout)
    R_vec_i = np.ones(Nin*Nout + Nin + Nout)
    State = Network_State(Variabs)
    State.initiate_resistances(BigClass, R_vec_i)
    State.initiate_accuracy_vec(BigClass, measure_accuracy_every)
    BigClass.add_State(State)  # add to big class
    
    return Variabs, Strctr, State, BigClass


def build_NET_and_plot(BigClass, M):
    ## build network graphics class and plot structure
    NET = Networkx_Net(50.0, 0.01)  # scale and squish
    NET.buildNetwork(BigClass)
    NET.build_pos_lattice(BigClass, plot=False)
    BigClass.add_NET(NET)  # add to big class
    NET.save_R_reordered(BigClass.State.R_in_t[-1], BigClass.Strctr.EIEJ_plots)
    NET.save_p_reordered(BigClass.State.p)
    NET.save_u_reordered(BigClass.State.u, BigClass.Strctr.EIEJ_plots)
    plot_functions.plot_importants(BigClass, M, movmean_loss=True, include_network=True, node_labels=False)


def train_loop(Variabs, Strctr, State, BigClass, loss_type, calculate_cosine_sim):
    dK_arr = []
    cosine_vec = []
    loss_mean = [1, 1]
    for i in range(Variabs.iterations):    
        # staying stay_sample iteration under same sample
        if use_p_tag and noise_to_extra:
            k = 2*(i//stay_sample) + 1
            if not(i%4):
                k-=1
        elif use_p_tag and not(noise_to_extra):
            k = (i//stay_sample)*2 + i%2
        elif not(use_p_tag) and noise_to_extra:
            k = (i//stay_sample)
        else:
            k = (i//stay_sample)
        # print('k', k)

        for l in range(1):  # Repeat the block l times
            # # measure modality
            # draw input and desired outputs from dataset and solve flow
            if not((i+1) % 4):  # add noise only at i=3 etc.
                State.draw_p_in_and_desired(Variabs, k, noise_to_extra=noise_to_extra)  # noise to extra nodes every 2nd iteration
                State.solve_flow_given_modality(BigClass, "measure", noise_to_extra=noise_to_extra)  # measure, don't change R's
            else:  # dont add noise to extra nodes
                State.draw_p_in_and_desired(Variabs, k)
                State.solve_flow_given_modality(BigClass, "measure")
                
            if calculate_cosine_sim or R_update == 'grad_desc':
                # GD cost
                State.calc_GD_cost(Strctr, State.desired, mod='measure', func='MSE')

                # change in K grad desc
                delta_K_GD = State.dK_grad_desc(Strctr, dK_step, State.desired, func='MSE')   

            if not i % 2 and use_p_tag:  # even iterations, take another sampled pressure and measure again
                pass
            else:  # odd iterations, go to update modality and update resistances
                if net_type == 'beads':  # if beads - need a few iterations to converge
                    update_iters = 6
                else:  # if not beads, converges within a single iteration
                    update_iters = 1

                State.t += 1
                State.calc_loss(BigClass)
                loss_mean = np.mean(np.abs(State.loss), axis=1)

                if not((i + 1) % 4) and access_interNodes:
                    State.update_inter(BigClass)
                else:
                    if training_scheme == 'GD_like':
                        State.calc_x_update_vec(BigClass)
                    State.update_input(BigClass)
                    State.update_output(BigClass)

                # # update modality
                for m in range(update_iters):
                    # measure and don't change resistances
                    State.solve_flow_given_modality(BigClass, "update", access_inters=access_interNodes)
                    # print('Power', np.sum(State.u**2*State.R_in_t[-1]))

                    # comment out update of R's
                    State.update_Rs(BigClass)
                    # State.update_Rs(BigClass, delta_K)
                    
                    if calculate_cosine_sim:
                        # dot product my scheme and grad desc
                        State.dK = statistics.dK(State.R_in_t)
                        # print(State.R_in_t)
                        dK_arr.append(State.dK)
                        # print('dK my way', State.dK)
                        cosine_sim = statistics.cosine_similarity(State.dK[:-Nout], delta_K_GD[:-Nout])
                        # print('cosine similarity', cosine_sim)
                        cosine_vec.append(cosine_sim) 
                    
                # # anneal learning rate
                if anneal and State.t<iterations:
                    State.update_alpha(BigClass, T_annealing)
                    
    # scalar loss in time
    if task_type == 'Regression':
        denominator_dataset = Variabs.y_train
    elif task_type == 'Iris_classification':
        denominator_dataset = array([[np.mean(State.targets_mat)]])
    if loss_type == 'MAE':
        State.loss_scalar_in_t = np.mean(np.mean(np.abs(State.loss_in_t), axis=1), axis=1)
        State.loss_scalar_in_t = State.loss_scalar_in_t / np.mean(np.abs(denominator_dataset), axis=1)
    elif loss_type == 'MSE':
        State.loss_scalar_in_t = np.mean(np.mean(np.square(State.loss_in_t), axis=1), axis=1)
        State.loss_scalar_in_t = State.loss_scalar_in_t / np.mean(np.square(denominator_dataset), axis=1)
        
    if calculate_cosine_sim:
        State.cosine_mean = np.nanmean(cosine_vec)

    return State

# Loop hysteresis

In [ ]:
window_for_mean = 160
start = 1
end = 10
random_state_M_vec = array([42, 43, 44, 45, 46, 47, 48, 49, 50, 51]) 
# random_state_M_vec = array([42]) 
norm_mean_loss = np.zeros([end-start+1,end-start+1, np.shape(random_state_M_vec)[0]])
min_mean_loss = np.zeros([end-start+1,end-start+1, np.shape(random_state_M_vec)[0]])
cosine_mean = np.zeros([end-start+1,end-start+1, np.shape(random_state_M_vec)[0]])
# Nin_vec = np.linspace(start,end,end-start+1).astype(np.int32)
Nin_vec = np.array([2])
# Nout_vec = np.linspace(start,end,end-start+1).astype(np.int32)
Nout_vec = np.array([3])
print('Nin_vec ', Nin_vec)
print('Nout_vec ', Nout_vec)
# Nin_vec = np.array([2])
# Nout_vec = np.array([3])
alpha1: float = 0.95
# anneal = False
anneal = True
# T_annealing: float = 1000  # for cosine annealing scheme
T_annealing: float = 2.0  # for cosine annealing scheme
calculate_cosine_sim = False

for k, random_state_M in enumerate(random_state_M_vec):
    M_values = functions.random_gen_M(random_state_M, 10*10)
    for i, Nin in enumerate(Nin_vec):
        for j, Nout in enumerate(Nout_vec):
            alpha = alpha1
            print('iteration out of 8: ', k)
            print('Nin', Nin)
            print('Nout', Nout)
            Variabs, Strctr, State, BigClass = network_build_given_Nin_Nout(Nin, Nout, M_values)
            State = train_loop(Variabs, Strctr, State, BigClass, loss_type, calculate_cosine_sim)
            
            loss_movmean = statistics.mov_ave(State.loss_scalar_in_t, 400)
            min_mean_loss_ij = loss_movmean[loss_movmean == np.min(loss_movmean)]
            
            # if loss too big decrease alpha and calculate again
            if min_mean_loss_ij > 0.01:
                print('too big loss, decreasing alpha from alpha=', alpha)
                alpha = alpha / 2
                print('to new alpha=', alpha)
                Variabs, Strctr, State, BigClass = network_build_given_Nin_Nout(Nin, Nout, M_values)
                State = train_loop(Variabs, Strctr, State, BigClass, loss_type, calculate_cosine_sim)
                loss_movmean = statistics.mov_ave(State.loss_scalar_in_t, 400)
                min_mean_loss_ij = min(min_mean_loss_ij, loss_movmean[loss_movmean == np.min(loss_movmean)][0])
                print('min_mean_loss_ij', min_mean_loss_ij)
                if min_mean_loss_ij > 0.01:
                    print('still too big loss, increasing alpha from alpha=', alpha)
                    alpha = alpha * 4
                    print('to new alpha=', alpha)
                    Variabs, Strctr, State, BigClass = network_build_given_Nin_Nout(Nin, Nout, M_values)
                    State = train_loop(Variabs, Strctr, State, BigClass, loss_type, calculate_cosine_sim)
                    loss_movmean = statistics.mov_ave(State.loss_scalar_in_t, 400)
                    min_mean_loss_ij = min(min_mean_loss_ij, loss_movmean[loss_movmean == np.min(loss_movmean)][0])
                    print('min_mean_loss_ij', min_mean_loss_ij)
            print('min_mean_loss_ij', min_mean_loss_ij)
            # print('cosine_mean', State.cosine_mean)
            # build_NET_and_plot(BigClass, Variabs.M)
            # norm_mean_loss[i, j, k] = norm_mean_loss_ij
            min_mean_loss[i, j, k] = min_mean_loss_ij
            # cosine_mean[i, j ,k] = State.cosine_mean
            plot_functions.plot_importants(BigClass, movmean_loss = True, include_network=False, node_labels=False)

In [ ]:
min_mean_loss_hysteresis = min_mean_loss 
# min_mean_loss_no_hysteresis = min_mean_loss

In [ ]:
print(np.mean(min_mean_loss[0,0]))
print(np.std(min_mean_loss[0,0]))
print(min_mean_loss[0,0])
plt.violinplot([min_mean_loss_hysteresis[0,0], min_mean_loss_no_hysteresis[0,0]] , showmeans=True, showextrema=False, showmedians=False)

# Formatting
plt.xticks([1, 2], ["Hysteresis", "No hysteresis"])  # Label x-axis
plt.ylabel("Final Loss")
# plt.title("Violin plot of data distribution")
plt.yscale("log")
plt.show()

In [ ]:
# Reload the module to reflect any changes made
importlib.reload(figure_plots)

figure_plots.loss_afo_in_out(min_mean_loss, min_mean_loss, Colorscheme)

## Loop multiple Nin Nout

In [ ]:
window_for_mean = 160
start = 1
end = 10
random_state_M_vec = array([42, 43, 44, 45]) 
# random_state_M_vec = array([42]) 
norm_mean_loss = np.zeros([end-start+1,end-start+1, np.shape(random_state_M_vec)[0]])
min_mean_loss = np.zeros([end-start+1,end-start+1, np.shape(random_state_M_vec)[0]])
cosine_mean = np.zeros([end-start+1,end-start+1, np.shape(random_state_M_vec)[0]])
# Nin_vec = np.linspace(start,end,end-start+1).astype(np.int32)
Nin_vec = np.array([2])
# Nout_vec = np.linspace(start,end,end-start+1).astype(np.int32)
Nout_vec = np.array([3])
print('Nin_vec ', Nin_vec)
print('Nout_vec ', Nout_vec)
alpha1: float = 0.136
# anneal = False
anneal = True
# T_annealing: float = 1000  # for cosine annealing scheme
T_annealing: float = 2.0  # for cosine annealing scheme
calculate_cosine_sim = True

for k, random_state_M in enumerate(random_state_M_vec):
    M_values = functions.random_gen_M(random_state_M, 10*10)
    for i, Nin in enumerate(Nin_vec):
        for j, Nout in enumerate(Nout_vec):
            # alpha: float = copy.copy(alpha1)*(Nin+Nout)
            if Nin==1:
                T_annealing = 0.5  # for cosine annealing scheme
                if Nout == 3:
                    alpha = 0.4
                elif Nout == 6:
                    alpha = 0.55
                elif Nout == 7:
                    alpha = 0.65
                elif Nout == 8:
                    alpha = 0.75
                elif Nout == 9:
                    alpha = 0.78
                elif Nout == 10:
                    alpha = 0.78
                else:
                    alpha = 0.1 * Nout
            elif Nin==2:
                T_annealing = 0.5  # for cosine annealing scheme
                if Nout==1:
                    alpha=0.2
                elif Nout==2:
                    alpha = 1.1
                elif Nout==3:
                    alpha=0.55
                elif Nout==4:
                    alpha=1.0
                elif Nout==5:
                    alpha=1.25
                elif Nout==6:
                    alpha = 1.6
                elif Nout==7:
                    alpha=1.8
                elif Nout==8:
                    alpha=2.1 
                elif Nout==9:
                    alpha=2.4
                elif Nout==10:
                    alpha=2.9
            elif Nout==1:
                T_annealing = 0.5  # for cosine annealing scheme
                if Nin == 3:
                    alpha=0.2
                if Nin>3:
                    alpha=0.1*Nin
            elif Nout==2:
                T_annealing = 0.5  # for cosine annealing scheme
                if Nin==3:
                    alpha=0.5
                elif Nin==4:
                    alpha=1.1
                elif Nin==5:
                    alpha=1.3
                elif Nin==6:
                    alpha=1.3
                elif Nin==7:
                    alpha=1.55
                elif Nin==8:
                    alpha=1.7
                elif Nin==9:
                    alpha=1.6
            elif Nout==3:
                T_annealing = 0.5  # for cosine annealing scheme
                if Nin==2:
                    alpha=0.85 
                elif Nin==3:
                    alpha=0.85
                elif Nin==4:
                    alpha=0.85
                elif Nin==5:
                    alpha=1.0
                elif Nin==6:
                    alpha=1.15
                elif Nin==7:
                    alpha=1.3
                elif Nin==8:
                    alpha=1.5
                elif Nout==9:
                    alpha=1.7
                elif Nout==10:
                    alpha=3.0
            elif Nout==1:
                alpha = 0.06
            elif Nin<3 or Nout<3:
                T_annealing = 2
                alpha = 0.45
            else:
                T_annealing = 2
                alpha = 1.4
            print('iteration out of 8: ', k)
            print('Nin', Nin)
            print('Nout', Nout)
            Variabs, Strctr, State, BigClass = network_build_given_Nin_Nout(Nin, Nout, M_values)
            State = train_loop(Variabs, Strctr, State, BigClass, loss_type, calculate_cosine_sim)
            
            loss_movmean = statistics.mov_ave(State.loss_scalar_in_t, 400)
            min_mean_loss_ij = loss_movmean[loss_movmean == np.min(loss_movmean)]
            
            # if loss too big decrease alpha and calculate again
            if min_mean_loss_ij > 0.01:
                print('too big loss, decreasing alpha from alpha=', alpha)
                alpha = alpha / 4
                print('to new alpha=', alpha)
                Variabs, Strctr, State, BigClass = network_build_given_Nin_Nout(Nin, Nout, M_values)
                State = train_loop(Variabs, Strctr, State, BigClass, loss_type, calculate_cosine_sim)
                loss_movmean = statistics.mov_ave(State.loss_scalar_in_t, 400)
                min_mean_loss_ij = min(min_mean_loss_ij, loss_movmean[loss_movmean == np.min(loss_movmean)][0])
                print('min_mean_loss_ij', min_mean_loss_ij)
                if min_mean_loss_ij > 0.01:
                    print('still too big loss, increasing alpha from alpha=', alpha)
                    alpha = alpha * 8
                    print('to new alpha=', alpha)
                    Variabs, Strctr, State, BigClass = network_build_given_Nin_Nout(Nin, Nout, M_values)
                    State = train_loop(Variabs, Strctr, State, BigClass, loss_type, calculate_cosine_sim)
                    loss_movmean = statistics.mov_ave(State.loss_scalar_in_t, 400)
                    min_mean_loss_ij = min(min_mean_loss_ij, loss_movmean[loss_movmean == np.min(loss_movmean)][0])
                    print('min_mean_loss_ij', min_mean_loss_ij)
            print('min_mean_loss_ij', min_mean_loss_ij)
            print('cosine_mean', State.cosine_mean)
            # build_NET_and_plot(BigClass, Variabs.M)
            # norm_mean_loss[i, j, k] = norm_mean_loss_ij
            min_mean_loss[i, j, k] = min_mean_loss_ij
            cosine_mean[i, j ,k] = State.cosine_mean
            plot_functions.plot_importants(BigClass, movmean_loss = True, include_network=False, node_labels=False)

In [ ]:
# loss_mat_mean_nonlin = np.mean(norm_mean_loss, axis=2)
# print(loss_mat_mean_nonlin)

In [ ]:
save_folder_prelim = 'C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/Network combine/Network_combine/'
# save_folder_prelim = 'C:/Users/roiee/OneDrive - huji.ac.il/PhD/Network Simulation repo/Network combine/Network_combine/'

np.save(save_folder_prelim + 'loss_mat.npy', min_mean_loss)
np.save(save_folder_prelim + 'cos_mat.npy', cosine_mean[:,:,:4])

In [ ]:
cos_mat_expand_expand = np.load('C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2025.7/cos_sim_lin_nonlin/cos_mat_expand_expand_lin.npy')

In [ ]:
# cos_mat_expand_expand[0,:,:] = cosine_mean[0,:,:]
np.save(save_folder_prelim + 'cos_mat_expand_expand.npy', cos_mat_expand_expand)

In [ ]:
load_folder = 'C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.10/loss_afo_NinNout_8runs_10x10_normalizeThroughLinept75/'

loss_mat = np.load(load_folder + 'loss_mat.npy')
cos_mat_expand = np.load('C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2025.7/cos_sim_lin_nonlin/cos_mat_expand_lin.npy')

new_slice = cosine_mean[:,0,:]
new_slice_reshaped = new_slice[:, np.newaxis, :]
cos_mat_expand_expand = np.concatenate((new_slice_reshaped, cos_mat_expand), axis=1)
print(cos_mat_expand_expand)


In [ ]:
# print(norm_mean_loss)
# print(np.mean(norm_mean_loss, axis=2))
# np.std(norm_mean_loss, axis=2)

# norm_mean_loss with markers and no line
plt.plot(norm_mean_loss[1, :], marker='o', linestyle='None')  # 'o' for circle markers, 'None' for no line

# Set y-axis to log scale
plt.yscale('log')

plt.legend(['Loss at final step'], loc='lower right')

# Set y-ticks every 1 unit (for log scale, this will display log-spaced ticks)
plt.gca().xaxis.set_major_locator(plt.MultipleLocator(1))

# Display the plot
plt.show()